<a href="https://colab.research.google.com/github/ThinkingBeyond/BeyondAI-2025/blob/main/Aryan%20Basnet%2C%20Arnav%20Maharjan%20and%20Ashila%20A%20M%20Ardiyansyah/02_dataset2_HIC_TB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NOTE ON RUNTIME AND OUTPUTS**

# Google Colab may have disconnected or reset the runtime during long training sessions, crashes, or memory interruptions. When this occurred, some previously displayed outputs in the notebook were no longer visible. However, all results remained saved and logged correctly. Each model’s complete metrics and metadata were stored as JSON files in my Google Drive folder:

# [https://drive.google.com/drive/folders/1ejlJaZhHEBm-1khLBJ--mbG2pg5TZHoJ?usp=sharing](https://drive.google.com/drive/folders/1ejlJaZhHEBm-1khLBJ--mbG2pg5TZHoJ?usp=sharing)

# These JSON files contain the full and reliable outputs for all models across all datasets, even if certain notebook outputs were lost due to runtime resets.

# ==============================
# SETUP: Freeze all package versions
# ==============================
Ensure reproducibility by installing the exact versions of packages used in these notebooks. This includes pre-installed packages in Colab.

The packages and versions used are:

- numpy==1.25.2
- pandas==2.1.1
- matplotlib==3.8.0
- seaborn==0.12.2
- scikit-learn==1.3.2
- tensorflow==2.15.0
- keras==2.15.0
- scipy==1.11.2
- opencv-python==4.9.0.73
- Pillow==10.0.1
- h5py==3.9.0
- google-colab==2.0.0

# THIS DATASET WAS INITIALLY MISTAKENLY LABELED AS "LMIC". IT'S ACTUALLY FROM A "HIC" AND THIS HAS BEEN CORRECTED IN OUR FINAL OUTCOME.

In [ ]:
# ==========================================
# CHEST X-RAY CLASSIFICATION - DATASET 2
# TUBERCULOSIS DATABASE
# ==========================================

# STEP 1: METADATA
# Defining dataset name, source, income level, and number of classes
DATASET_NAME = "dataset2_tb_chest_xray"
COUNTRY_INCOME_LEVEL = "LMIC"  # Low-Middle Income Country
DATASET_SOURCE = "https://www.kaggle.com/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset"
NUM_CLASSES = 2  # Normal and TB

# Printing basic dataset info for traceability
print(f"Dataset: {DATASET_NAME}")
print(f"Income Level: {COUNTRY_INCOME_LEVEL}")
print(f"Classes: Normal, Tuberculosis")

# STEP 2: MOUNT DRIVE
# Mounting Google Drive to save results
from google.colab import drive
drive.mount('/content/drive')

# Creating results folder if not already present
!mkdir -p /content/drive/MyDrive/xray_research_results

# STEP 3: IMPORT DATASET
# Importing libraries for dataset handling, model building, evaluation, and plotting
import kagglehub
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import json
import time
import gc
import matplotlib.pyplot as plt
import warnings
import shutil
warnings.filterwarnings('ignore')

# Downloading dataset
print("Downloading dataset...")
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
print("Path to dataset files:", path)

# Exploring dataset folder structure
print("\nDataset structure:")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Showing first 5 files only
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files)-5} more files")

# STEP 4: ORGANIZE DATA INTO TRAIN/VAL/TEST SPLITS
# Creating train/val/test folders manually since dataset doesn't have predefined splits
print("\nOrganizing data into train/val/test splits...")

# Creating folder structure for each split and class
base_dir = '/content/organized_data'
for split in ['train', 'val', 'test']:
    for class_name in ['Normal', 'TB']:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

# Function to copy images into train/val/test splits
# Shuffling and splitting images by class
def organize_dataset(source_path, base_dir, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
    """
    Organizing TB dataset into train/val/test splits
    """
    tb_images = []
    normal_images = []

    # Collecting images by folder name
    for root, dirs, files in os.walk(source_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                file_path = os.path.join(root, file)
                parent_folder = os.path.basename(root)
                if parent_folder.lower() == 'tuberculosis':
                    tb_images.append(file_path)
                elif parent_folder.lower() == 'normal':
                    normal_images.append(file_path)

    print(f"Found {len(tb_images)} TB images")
    print(f"Found {len(normal_images)} Normal images")

    # Splitting and copying files to folders
    for class_images, class_name in [(tb_images, 'TB'), (normal_images, 'Normal')]:
        np.random.seed(42)  # For reproducibility
        np.random.shuffle(class_images)

        n = len(class_images)
        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)

        train_files = class_images[:train_end]
        val_files = class_images[train_end:val_end]
        test_files = class_images[val_end:]

        print(f"\n{class_name} split: Train={len(train_files)}, Val={len(val_files)}, Test={len(test_files)}")

        # Copying files to organized folders
        for files, split in [(train_files, 'train'), (val_files, 'val'), (test_files, 'test')]:
            dest_dir = os.path.join(base_dir, split, class_name)
            for i, src in enumerate(files):
                dest = os.path.join(dest_dir, f"{class_name}_{split}_{i}{os.path.splitext(src)[1]}")
                shutil.copy2(src, dest)

# Running the organization function
organize_dataset(path, base_dir)

# STEP 5: SETUP DATA PREPROCESSING
# Setting image size, batch size, epochs, class mapping
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

CLASS_NAMES = ['Normal', 'TB']
CLASS_MAPPING = {
    'Normal': 0,   # Healthy
    'TB': 1        # Tuberculosis
}

# Defining augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# No augmentation for validation/test
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Paths for generators
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Verifying directory existence
print("\nChecking directories:")
print(f"Train dir exists: {os.path.exists(train_dir)}")
print(f"Val dir exists: {os.path.exists(val_dir)}")
print(f"Test dir exists: {os.path.exists(test_dir)}")

# Creating data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

validation_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# Printing dataset info
print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")
print(f"Classes found: {train_generator.class_indices}")

# STEP 6-9: MODEL DEFINITIONS, TRAINING, EVALUATION
# (Same as Dataset 1: baseline CNN, transfer models, training function, saving JSON)
# Add a runtime note here if desired, pointing to saved JSON folder


In [ ]:
# ==========================================
# RETRAIN EFFICIENTNETB0 AND RESNET50 ONLY
# FOR DATASET 2 (TB)
# ==========================================

# STEP 1: METADATA
DATASET_NAME = "dataset2_tb_chest_xray"
COUNTRY_INCOME_LEVEL = "LMIC"
NUM_CLASSES = 2
CLASS_NAMES = ['Normal', 'TB']

print(f"Retraining EfficientNetB0 and ResNet50 for {DATASET_NAME}")

# STEP 2: IMPORTS
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
import json
import time
import gc
import warnings
warnings.filterwarnings('ignore')

# STEP 3: SETUP DATA (assuming organized_data already exists from previous run)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

base_dir = '/content/organized_data'

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

validation_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nData loaded:")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

# STEP 4: MODEL FUNCTIONS
def create_transfer_model(base_model_name, input_shape=(224, 224, 3), num_classes=2):
    """Create transfer learning model"""
    if base_model_name == 'EfficientNetB0':
        base = tf.keras.applications.EfficientNetB0(
            input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'ResNet50':
        base = tf.keras.applications.ResNet50(
            input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown model: {base_model_name}")

    # Freeze base layers
    base.trainable = False

    # Add custom head
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    return model

def train_and_evaluate(model, model_name, train_gen, val_gen, test_gen):
    """Train model and return results"""
    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")

    # Compile
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    )

    # Train
    start_time = time.time()
    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    training_time = (time.time() - start_time) / 60

    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)

    # Get predictions
    predictions = model.predict(test_gen)
    y_pred = (predictions > 0.5).astype(int).flatten()
    y_true = test_gen.classes

    # Calculate metrics
    f1_per_class = [
        f1_score(y_true == 0, y_pred == 0),  # F1 for Normal
        f1_score(y_true == 1, y_pred == 1)   # F1 for TB
    ]
    f1_weighted = f1_score(y_true, y_pred, average='weighted')

    cm = confusion_matrix(y_true, y_pred)

    # Prepare results
    results = {
        'dataset_name': DATASET_NAME,
        'country_income': COUNTRY_INCOME_LEVEL,
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'f1_per_class': f1_per_class,
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
        'training_time_minutes': float(training_time),
        'num_images_train': train_gen.samples,
        'num_images_val': val_gen.samples,
        'num_images_test': test_gen.samples,
        'num_parameters': int(model.count_params()),
        'test_accuracy': float(test_acc),
        'epochs_trained': len(history.history['loss'])
    }

    # Print results
    print(f"\n{model_name} Results:")
    print(f"F1 Scores per class: {f1_per_class}")
    print(f"Weighted F1 Score: {f1_weighted:.4f}")
    print(f"Training time: {training_time:.2f} minutes")
    print(f"Parameters: {model.count_params():,}")

    # Save results
    filename = f'/content/drive/MyDrive/xray_research_results/{DATASET_NAME}_{COUNTRY_INCOME_LEVEL}_{model_name}_results.json'
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {filename}")

    # Clear memory
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return results

# STEP 5: TRAIN MODELS
all_results = []

# Model 1: EfficientNetB0 (with learning rate reduction)
model = create_transfer_model('EfficientNetB0', num_classes=NUM_CLASSES)
results = train_and_evaluate(model, 'EfficientNetB0', train_generator, validation_generator, test_generator)
all_results.append(results)

# Model 2: ResNet50
model = create_transfer_model('ResNet50', num_classes=NUM_CLASSES)
results = train_and_evaluate(model, 'ResNet50', train_generator, validation_generator, test_generator)
all_results.append(results)

# STEP 6: SUMMARY
print("\n" + "="*60)
print("RETRAINING COMPLETE - SUMMARY")
print("="*60)

for result in all_results:
    print(f"\n{result['model_name']}:")
    print(f"  Weighted F1: {result['f1_weighted']:.4f}")
    print(f"  Training Time: {result['training_time_minutes']:.2f} min")
    print(f"  Parameters: {result['num_parameters']:,}")

Retraining EfficientNetB0 and ResNet50 for dataset2_tb_chest_xray
Found 2939 images belonging to 2 classes.
Found 630 images belonging to 2 classes.
Found 631 images belonging to 2 classes.

Data loaded:
Training samples: 2939
Validation samples: 630
Test samples: 631
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

Training EfficientNetB0
Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 356s 4s/step - accuracy: 0.8293 - loss: 0.4859 - val_accuracy: 0.8333 - val_loss: 0.4515 - learning_rate: 0.0010
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 339s 4s/step - accuracy: 0.8376 - loss: 0.4561 - val_accuracy: 0.8333 - val_loss: 0.4544 - learning_rate: 0.0010
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 339s 4s/step - accuracy: 0.8358 - loss: 0.4683 - val_accuracy: 0.8333 - val_loss: 0.4505 - learning_rate: 0.0010
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 345s 4s/step - accuracy: 0.8248 - loss: 0.4808 - val_accuracy: 0.8333 - val_loss: 0.4505 - learning_rate: 0.0010
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 366s 4s/step 

KeyboardInterrupt: 

In [ ]:
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset -p /content
!unzip -q /content/tuberculosis-tb-chest-xray-dataset.zip -d /content/dataset2_raw

Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset
License(s): copyright-authors
 97% 642M/663M [00:08<00:00, 141MB/s]
100% 663M/663M [00:08<00:00, 85.7MB/s]


In [ ]:
import os
import shutil
import numpy as np

# Setting paths for raw dataset and cleaned, organized dataset
RAW_DIR = "/content/dataset2_raw/TB_Chest_Radiography_Database"  # pointing to unzipped raw dataset folder
BASE_DIR = "/content/dataset2_cleaned"  # will store organized train/val/test splits

# Defining dataset split ratios
train_ratio = 0.7  # using 70% of images for training
val_ratio = 0.15   # using 15% of images for validation
test_ratio = 0.15  # using 15% of images for testing

def organize_dataset(source_path, base_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    # Creating base directory if it does not exist
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)

    # Detecting class folders automatically
    CLASSES = [d for d in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, d))]
    print(f"Detected classes: {CLASSES}")

    # Creating train/val/test folders for each class
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir):
            os.makedirs(split_dir)
        for cls in CLASSES:
            cls_split_dir = os.path.join(split_dir, cls)
            if not os.path.exists(cls_split_dir):
                os.makedirs(cls_split_dir)

    # Iterating through each class folder to shuffle and split images
    for cls in CLASSES:
        cls_path = os.path.join(source_path, cls)
        # Collecting all image files for the class
        images = [os.path.join(cls_path, f) for f in os.listdir(cls_path)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        np.random.seed(42)  # setting seed for reproducibility
        np.random.shuffle(images)  # shuffling images

        # Calculating number of images for each split
        n_total = len(images)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        # Assigning images to train/val/test splits
        splits = {
            'train': images[:n_train],
            'val': images[n_train:n_train + n_val],
            'test': images[n_train + n_val:]
        }

        # Copying images into respective directories with new filenames
        for split, split_images in splits.items():
            dest_dir = os.path.join(base_dir, split, cls)
            for i, src in enumerate(split_images):
                shutil.copy2(src, os.path.join(dest_dir, f"{cls}_{i}{os.path.splitext(src)[1]}"))

    print("Dataset organized successfully!")  # confirming dataset organization

# Running the function to organize the dataset
organize_dataset(RAW_DIR, BASE_DIR)

Detected classes: ['Tuberculosis', 'Normal']
Dataset organized successfully!


In [ ]:
# Path to the unzipped raw dataset
RAW_DIR = "/content/dataset2_raw/TB_Chest_Radiography_Database"

# Path where the organized dataset will go
BASE_DIR = "/content/dataset2_cleaned"

# Check that the directories exist
import os
print("RAW_DIR exists:", os.path.exists(RAW_DIR))
print("BASE_DIR exists:", os.path.exists(BASE_DIR))

RAW_DIR exists: True
BASE_DIR exists: True


In [ ]:
# List the contents of RAW_DIR
import os
print("Contents of RAW_DIR:", os.listdir(RAW_DIR))

Contents of RAW_DIR: ['README.md.txt', 'Tuberculosis', 'Normal.metadata.xlsx', 'Normal', 'Tuberculosis.metadata.xlsx']


In [ ]:
# Check first 5 images in each class
for cls in ['Normal', 'Tuberculosis']:
    cls_path = os.path.join(RAW_DIR, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"{cls} images (first 5):", images[:5])

Normal images (first 5): ['Normal-2741.png', 'Normal-528.png', 'Normal-1362.png', 'Normal-901.png', 'Normal-860.png']
Tuberculosis images (first 5): ['Tuberculosis-693.png', 'Tuberculosis-251.png', 'Tuberculosis-119.png', 'Tuberculosis-601.png', 'Tuberculosis-656.png']


In [ ]:
import shutil
import numpy as np

# Split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

np.random.seed(42)  # for reproducibility

for cls in ['Normal', 'Tuberculosis']:
    cls_path = os.path.join(RAW_DIR, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    np.random.shuffle(images)

    n_total = len(images)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train+n_val]
    test_imgs = images[n_train+n_val:]

    # Move images
    for img in train_imgs:
        shutil.copy2(os.path.join(cls_path, img), os.path.join(BASE_DIR, 'train', cls, img))
    for img in val_imgs:
        shutil.copy2(os.path.join(cls_path, img), os.path.join(BASE_DIR, 'val', cls, img))
    for img in test_imgs:
        shutil.copy2(os.path.join(cls_path, img), os.path.join(BASE_DIR, 'test', cls, img))

print("Images organized into train/val/test successfully.")

Images organized into train/val/test successfully.


In [ ]:
import os
import shutil
import hashlib
import numpy as np

# --- PATHS ---
RAW_DIR = "/content/dataset2_raw/TB_Chest_Radiography_Database"
BASE_DIR = "/content/dataset2"

CLASSES = ["Normal", "Tuberculosis"]

# --- FUNCTION TO HASH IMAGES ---
def hash_image(file_path):
    """Return a hash for an image file to detect duplicates."""
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# --- REMOVE DUPLICATES ---
def remove_duplicates(folder_path):
    for cls in CLASSES:
        cls_path = os.path.join(folder_path, cls)
        hashes = {}
        duplicates = []

        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                fpath = os.path.join(cls_path, fname)
                h = hash_image(fpath)
                if h in hashes:
                    duplicates.append(fpath)
                else:
                    hashes[h] = fpath

        for dup in duplicates:
            os.remove(dup)

        print(f"Removed {len(duplicates)} duplicates from {cls_path}")

# --- ORGANIZE DATASET ---
def organize_dataset(source_path, base_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            os.makedirs(os.path.join(base_dir, split, cls), exist_ok=True)

    for cls in CLASSES:
        cls_path = os.path.join(source_path, cls)
        images = [os.path.join(cls_path, f) for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        np.random.seed(42)
        np.random.shuffle(images)

        n_total = len(images)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        n_test = n_total - n_train - n_val

        splits = {
            "train": images[:n_train],
            "val": images[n_train:n_train+n_val],
            "test": images[n_train+n_val:]
        }

        for split, split_images in splits.items():
            dest_dir = os.path.join(base_dir, split, cls)
            for i, src in enumerate(split_images):
                shutil.copy2(src, os.path.join(dest_dir, f"{cls}_{i}{os.path.splitext(src)[1]}"))

    print("Images organized into train/val/test successfully.")

# --- RUN CLEANUP AND ORGANIZATION ---
remove_duplicates(RAW_DIR)
organize_dataset(RAW_DIR, BASE_DIR)

Removed 0 duplicates from /content/dataset2_raw/TB_Chest_Radiography_Database/Normal
Removed 3 duplicates from /content/dataset2_raw/TB_Chest_Radiography_Database/Tuberculosis
Images organized into train/val/test successfully.


In [ ]:
import os
import random
import shutil

BASE_DIR = '/content/organized_data'
CLASSES = ['Normal', 'Tuberculosis']
TARGET_TOTAL = 4200

# Function to balance classes
def rebalance_dataset(base_dir, target_total=4200):
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'val')
    test_dir = os.path.join(base_dir, 'test')

    # Get counts
    counts = {}
    for split in ['train', 'val', 'test']:
        counts[split] = {}
        for cls in CLASSES:
            cls_dir = os.path.join(base_dir, split, cls)
            counts[split][cls] = len(os.listdir(cls_dir))

    print("Current counts per split:", counts)

    # Calculate total TB and Normal images
    total_tb = sum(counts[s]['Tuberculosis'] for s in counts)
    total_normal = sum(counts[s]['Normal'] for s in counts)

    # Target Normal = total images - total TB
    target_normal = target_total - total_tb
    print(f"Target Normal images: {target_normal}, TB images kept: {total_tb}")

    # Downsample Normal proportionally in each split
    for split in ['train', 'val', 'test']:
        cls_dir = os.path.join(base_dir, split, 'Normal')
        files = os.listdir(cls_dir)
        n_keep = int(len(files) * target_normal / total_normal)
        files_to_remove = random.sample(files, len(files) - n_keep)
        for f in files_to_remove:
            os.remove(os.path.join(cls_dir, f))
        print(f"{split}: Removed {len(files_to_remove)} Normal images")

rebalance_dataset(BASE_DIR)

Current counts per split: {'train': {'Normal': 4900, 'Tuberculosis': 0}, 'val': {'Normal': 1050, 'Tuberculosis': 0}, 'test': {'Normal': 1050, 'Tuberculosis': 0}}
Target Normal images: 4200, TB images kept: 0
train: Removed 1960 Normal images
val: Removed 420 Normal images
test: Removed 420 Normal images


In [ ]:
# ==========================================
# RETRAIN EFFICIENTNETB0 AND RESNET50 ONLY
# FOR DATASET 2 (TB) - USING CLASS WEIGHTS
# ==========================================

# STEP 1: METADATA
DATASET_NAME = "dataset2_tb_chest_xray"
COUNTRY_INCOME_LEVEL = "LMIC"  # Low-Middle Income Country
NUM_CLASSES = 2
CLASS_NAMES = ['Normal', 'TB']

print(f"Retraining EfficientNetB0 and ResNet50 for {DATASET_NAME}")

# STEP 2: IMPORT REQUIRED LIBRARIES
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import json
import time
import gc
import warnings
warnings.filterwarnings('ignore')

# STEP 3: DATASET PREPARATION
IMG_SIZE = (224, 224)  # standard input size for transfer models
BATCH_SIZE = 32
EPOCHS = 20

base_dir = '/content/organized_data'  # already organized train/val/test folders

# Data augmentation for training set to improve generalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# No augmentation for validation and test sets, only rescaling
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Setting up directories for generators
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Creating image generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

validation_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nData loaded:")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

# STEP 4: COMPUTE CLASS WEIGHTS
# Important for imbalanced datasets to avoid bias toward majority class
y_train = train_generator.classes
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {i: w for i, w in enumerate(class_weights_array)}

print(f"\nClass distribution in training:")
for i, cls_name in enumerate(CLASS_NAMES):
    print(f"  {cls_name} ({i}): {np.sum(y_train == i)} samples")
print(f"Class weights: {class_weights}")

# STEP 5: DEFINE TRANSFER LEARNING MODELS
def create_transfer_model(base_model_name, input_shape=(224, 224, 3), num_classes=2):
    """Create EfficientNetB0 or ResNet50 with custom classification head"""
    if base_model_name == 'EfficientNetB0':
        base = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'ResNet50':
        base = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown model: {base_model_name}")

    base.trainable = False  # freeze base layers to use pretrained features
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # binary classification
    return keras.Model(inputs, outputs)

# STEP 5b: TRAINING AND EVALUATION FUNCTION
def train_and_evaluate(model, model_name, train_gen, val_gen, test_gen, class_weights):
    """Train the model and compute metrics, saving results to JSON"""
    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")

    # Compile model with binary crossentropy loss
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Callbacks: early stopping + learning rate reduction
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

    # Start training
    start_time = time.time()
    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stop, reduce_lr],
        class_weight=class_weights,
        verbose=1
    )
    training_time = (time.time() - start_time)/60  # convert seconds to minutes

    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    predictions = model.predict(test_gen)
    y_pred = (predictions > 0.5).astype(int).flatten()
    y_true = test_gen.classes

    # Compute per-class F1 scores and weighted F1
    f1_per_class = [
        f1_score(y_true == 0, y_pred == 0),
        f1_score(y_true == 1, y_pred == 1)
    ]
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    # Prepare results dictionary
    results = {
        'dataset_name': DATASET_NAME,
        'country_income': COUNTRY_INCOME_LEVEL,
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'f1_per_class': f1_per_class,
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
        'training_time_minutes': float(training_time),
        'num_images_train': train_gen.samples,
        'num_images_val': val_gen.samples,
        'num_images_test': test_gen.samples,
        'num_parameters': int(model.count_params()),
        'test_accuracy': float(test_acc),
        'epochs_trained': len(history.history['loss'])
    }

    # Print metrics
    print(f"\n{model_name} Results:")
    print(f"F1 per class: {f1_per_class}")
    print(f"Weighted F1: {f1_weighted:.4f}")
    print(f"Training time: {training_time:.2f} min")
    print(f"Parameters: {model.count_params():,}")

    # Ensure results folder exists and save JSON
    results_dir = '/content/drive/MyDrive/xray_research_results'
    os.makedirs(results_dir, exist_ok=True)
    filename = os.path.join(results_dir, f'{DATASET_NAME}_{COUNTRY_INCOME_LEVEL}_{model_name}_results.json')
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {filename}")

    # Clean up memory
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return results

# STEP 6: TRAIN THE MODELS
all_results = []

# EfficientNetB0
model = create_transfer_model('EfficientNetB0')
results = train_and_evaluate(model, 'EfficientNetB0', train_generator, validation_generator, test_generator, class_weights)
all_results.append(results)

# ResNet50
model = create_transfer_model('ResNet50')
results = train_and_evaluate(model, 'ResNet50', train_generator, validation_generator, test_generator, class_weights)
all_results.append(results)

# STEP 7: SUMMARY
print("\n" + "="*60)
print("RETRAINING COMPLETE - SUMMARY")
print("="*60)

for result in all_results:
    print(f"\n{result['model_name']}:")
    print(f"  Weighted F1: {result['f1_weighted']:.4f}")
    print(f"  F1 per class: {result['f1_per_class']}")
    print(f"  Training Time: {result['training_time_minutes']:.2f} min")
    print(f"  Parameters: {result['num_parameters']:,}")

Retraining EfficientNetB0 and ResNet50 for dataset2_tb_chest_xray
Found 3429 images belonging to 2 classes.
Found 735 images belonging to 2 classes.
Found 736 images belonging to 2 classes.

Data loaded:
Training samples: 3429
Validation samples: 735
Test samples: 736

Class distribution in training:
  Normal (0): 2940 samples
  TB (1): 489 samples
Class weights: {0: np.float64(0.5831632653061225), 1: np.float64(3.5061349693251533)}

Training EfficientNetB0
Epoch 1/20
108/108 ━━━━━━━━━━━━━━━━━━━━ 382s 3s/step - accuracy: 0.5030 - loss: 0.7173 - val_accuracy: 0.1429 - val_loss: 0.7347 - learning_rate: 0.0010
Epoch 2/20
108/108 ━━━━━━━━━━━━━━━━━━━━ 382s 3s/step - accuracy: 0.2668 - loss: 0.6978 - val_accuracy: 0.1429 - val_loss: 0.7076 - learning_rate: 0.0010
Epoch 3/20
108/108 ━━━━━━━━━━━━━━━━━━━━ 385s 3s/step - accuracy: 0.5028 - loss: 0.6908 - val_accuracy: 0.1429 - val_loss: 0.6956 - learning_rate: 0.0010
Epoch 4/20
108/108 ━━━━━━━━━━━━━━━━━━━━ 389s 4s/step - accuracy: 0.7090 - loss: